In [1]:
import os, sys, io, html, subprocess
from contextlib import redirect_stdout

# Onde está o projeto? O notebook procura sozinho, nesta ordem:
#   1) na pasta atual ou numa subpasta (ex.: estacionamento-py/);
#   2) clonando REPO_URL, se preenchido (ex.: "https://github.com/usuario/estacionamento-py");
#   3) no Google Colab, pedindo o upload do .zip do projeto (botão "Escolher arquivos").
REPO_URL = ""


def achar_projeto():
    for raiz, pastas, arquivos in os.walk("."):
        if raiz.count(os.sep) > 3:
            pastas[:] = []
            continue
        pastas[:] = [p for p in pastas if not p.startswith((".", "__")) and p != "sample_data"]
        if "validadores.py" in arquivos and "scanner.py" in arquivos:
            return raiz
    return None


pasta = achar_projeto()
if pasta is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    pasta = achar_projeto()
if pasta is None and "google.colab" in sys.modules:
    import zipfile
    from google.colab import files
    print("Envie o arquivo .zip do projeto (ou preencha REPO_URL e rode de novo):")
    for nome in files.upload():
        if nome.endswith(".zip"):
            zipfile.ZipFile(nome).extractall(".")
    pasta = achar_projeto()
if pasta is None:
    raise FileNotFoundError("Não achei o projeto (validadores.py). Abra este notebook dentro da pasta do "
                            "projeto, preencha REPO_URL com o link do GitHub ou, no Colab, envie o .zip.")
os.chdir(pasta)
sys.path.insert(0, os.getcwd())

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
except ImportError:
    raise ImportError("Falta o ipywidgets. Instale com:  pip install ipywidgets  (no Colab ele já vem instalado)")

import validadores
import cancela as cn
import conectado
import scanner as sc
from scanner import scanner as scan
from afd import bateria, pares_equivalentes
from validadores import (afd_placa, afd_hora, RE_PLACA, RE_HORA,
                         PLACA_ACEITAS, PLACA_REJEITADAS, HORA_ACEITAS, HORA_REJEITADAS)

print("Projeto carregado ✔ —", os.getcwd())

Projeto carregado ✔ — c:\Users\lucas\Downloads\estacionamento-py-com-notebook (1)\estacionamento-py


In [2]:
# ---------------------------------------------------------------- tema e funções de apoio
C = {"bg": "#1a1d23", "painel": "#23272f", "linha": "#30353f", "texto": "#e8e9ec",
     "fraco": "#8b919c", "amarelo": "#f5c518", "ok": "#3ecf8e", "ok_fundo": "#13332a",
     "erro": "#ff5c5c", "erro_fundo": "#3a1b1e", "azul": "#5aa9ff"}
MONO = "Consolas, Menlo, 'DejaVu Sans Mono', monospace"
UI = "'Segoe UI', Helvetica, Arial, sans-serif"


def esc(s):
    return html.escape(str(s))


def card(corpo, titulo=None):
    cab = ""
    if titulo:
        cab = (f'<div style="color:{C["fraco"]};font:600 11px {UI};letter-spacing:.06em;'
               f'text-transform:uppercase;margin-bottom:8px">{esc(titulo)}</div>')
    return (f'<div style="background:{C["bg"]};color:{C["texto"]};border:1px solid {C["linha"]};'
            f'border-radius:8px;padding:12px 16px;margin:6px 0;font-family:{UI}">{cab}{corpo}</div>')


def pre(texto, altura="260px"):
    return (f'<pre style="background:{C["painel"]};color:{C["texto"]};font:12px/1.5 {MONO};margin:0;'
            f'padding:10px;border-radius:6px;max-height:{altura};overflow:auto;white-space:pre">{esc(texto)}</pre>')


def capturar(func, *args):
    # roda uma função que usa print() e devolve (texto_impresso, retorno)
    buf = io.StringIO()
    with redirect_stdout(buf):
        retorno = func(*args)
    return buf.getvalue(), retorno


def chip(texto, cor):
    return (f'<span style="display:inline-block;margin:0 6px 4px 0;padding:1px 8px;border-radius:10px;'
            f'border:1px solid {cor};color:{cor};font:11px {MONO}">{esc(texto)}</span>')

In [3]:
# ---------------------------------------------------------------- aba 1: validadores
AFDS = {
    "Placa": {"afd": afd_placa, "regex": RE_PLACA, "aceitas": PLACA_ACEITAS, "rejeitadas": PLACA_REJEITADAS,
              "inicio": "BRA2E19", "img": "img/afd_placa.png",
              "exemplos": [("ABC-1234", True), ("BRA2E19", True), ("BRA-2E19", False),
                           ("ABC-123", False), ("ABC1234", False), ("", False)]},
    "Horário": {"afd": afd_hora, "regex": RE_HORA, "aceitas": HORA_ACEITAS, "rejeitadas": HORA_REJEITADAS,
                "inicio": "08:15", "img": "img/afd_horario.png",
                "exemplos": [("08:15", True), ("23:59", True), ("24:00", False),
                             ("12:5", False), ("8:15", False), ("", False)]},
}


def percorrer(afd, w):
    # devolve (caminho, morreu_em, aceita); caminho = estados visitados (com o inicial);
    # morreu_em = índice do caractere sem transição (ou None se leu a palavra toda)
    q = afd.inicial
    caminho = [q]
    for i, ch in enumerate(w):
        prox = afd.delta.get((q, afd.classe(ch)))
        if prox is None:
            return caminho, i, False
        q = prox
        caminho.append(q)
    return caminho, None, q in afd.finais


def html_resultado(afd, w, caminho, morreu, aceita):
    cor = C["ok"] if aceita else C["erro"]
    if aceita:
        motivo = f"terminou em {caminho[-1]} ∈ F"
    elif morreu is not None:
        cl = afd.classe(w[morreu])
        motivo = f"morreu no meio: δ({caminho[-1]}, {cl if cl else 'fora de Σ'}) indefinida em {w[morreu]!r} (posição {morreu})"
    else:
        extra = afd.descricao.get(caminho[-1], "")
        motivo = f"terminou em {caminho[-1]} ∉ F" + (f" ({extra})" if extra else "")
    veredito = "✔ ACEITA" if aceita else "✘ REJEITADA"

    caixas = ""
    if not w:
        caixas = f'<span style="color:{C["fraco"]};font:14px {MONO}">ε (palavra vazia)</span>'
    for i, ch in enumerate(w):
        if morreu is None or i < morreu:
            borda, txt, bg = C["amarelo"], C["amarelo"], "transparent"
        elif i == morreu:
            borda, txt, bg = C["erro"], C["erro"], C["erro_fundo"]
        else:
            borda, txt, bg = C["linha"], C["fraco"], "transparent"
        simbolo = "␣" if ch == " " else esc(ch)
        caixas += (f'<span style="display:inline-block;min-width:22px;text-align:center;margin:0 3px 3px 0;'
                   f'padding:3px 5px;border:1px solid {borda};color:{txt};background:{bg};font:15px {MONO}">{simbolo}</span>')

    # caminho percorrido: p0 —L→ p1 —L→ p2 ...
    bolas = ""
    for i, q in enumerate(caminho):
        ultimo = (i == len(caminho) - 1)
        borda, bg, txt = C["amarelo"], C["painel"], C["amarelo"]
        if ultimo and morreu is None:
            borda, bg, txt = (C["ok"], C["ok"], "#0b1f17") if aceita else (C["erro"], C["erro"], "#ffffff")
        estilo_final = "double 4px" if q in afd.finais else "solid 2px"
        bolas += (f'<span style="display:inline-block;min-width:30px;text-align:center;padding:5px 7px;'
                  f'border-radius:20px;border:{estilo_final} {borda};background:{bg};color:{txt};font:bold 12px {MONO}">{esc(q)}</span>')
        if i < len(w) and (i < len(caminho) - 1 or morreu is not None):
            rotulo = afd.classe(w[i]) or "fora de Σ"
            bolas += f'<span style="color:{C["fraco"]};font:11px {MONO};margin:0 4px">—{esc(rotulo)}→</span>'
    if morreu is not None:
        bolas += (f'<span style="display:inline-block;min-width:30px;text-align:center;padding:5px 7px;border-radius:20px;'
                  f'border:2px solid {C["erro"]};background:{C["erro_fundo"]};color:{C["erro"]};font:bold 12px {MONO}">✕</span>')

    corpo = (f'<div style="display:flex;align-items:center;gap:18px;flex-wrap:wrap">'
             f'<div style="color:{cor};font:bold 26px {UI}">{veredito}</div>'
             f'<div><div style="color:{cor};font:12px {UI};margin-bottom:6px">{esc(motivo)}</div>{caixas}</div></div>'
             f'<div style="margin-top:12px;color:{C["fraco"]};font:11px {UI}">CAMINHO NO {esc(afd.nome)} '
             f'(círculo duplo = estado final)</div><div style="margin-top:6px">{bolas}</div>')
    return card(corpo)


def painel_validadores():
    modo = widgets.ToggleButtons(options=list(AFDS), value="Placa")
    entrada = widgets.Text(value=AFDS["Placa"]["inicio"], placeholder="digite aqui",
                           layout=widgets.Layout(width="280px"))
    dica = widgets.HTML(f'<span style="font:12px {UI};color:#8b919c">exemplos (verde = aceita, vermelho = rejeita):</span>')
    chips = widgets.HBox(layout=widgets.Layout(flex_flow="row wrap"))
    resultado = widgets.HTML()
    detalhes = widgets.HTML()
    gaveta = widgets.Accordion(children=[widgets.HTML(), widgets.VBox()])
    gaveta.set_title(0, "Bateria AFD × RegEx (R3)")
    gaveta.set_title(1, "Desenho do AFD (Graphviz)")
    gaveta.selected_index = None

    def cfg():
        return AFDS[modo.value]

    def montar_chips():
        botoes = []
        for texto, ok in cfg()["exemplos"]:
            b = widgets.Button(description=texto if texto else "ε (vazia)",
                               button_style="success" if ok else "danger",
                               layout=widgets.Layout(width="auto"))
            b.on_click(lambda _b, t=texto: setattr(entrada, "value", t))
            botoes.append(b)
        chips.children = botoes

    def atualizar_gaveta():
        c = cfg()
        txt, _ = capturar(bateria, c["afd"], c["regex"], c["aceitas"], c["rejeitadas"])
        gaveta.children[0].value = card(pre(txt, "320px"))
        if os.path.exists(c["img"]):
            with open(c["img"], "rb") as f:
                img = widgets.Image(value=f.read(), format="png", layout=widgets.Layout(max_width="100%"))
            gaveta.children[1].children = [img]
        else:
            gaveta.children[1].children = [widgets.HTML("<i>imagem não encontrada em img/</i>")]

    def atualizar(*_):
        c = cfg()
        afd, w = c["afd"], entrada.value
        caminho, morreu, aceita = percorrer(afd, w)
        resultado.value = html_resultado(afd, w, caminho, morreu, aceita)
        _, linhas = afd.rodar(w, imprimir=False)
        tabela_txt, _ = capturar(afd.tabela)
        detalhes.value = (
            '<div style="display:flex;gap:12px;flex-wrap:wrap">'
            f'<div style="flex:1;min-width:320px">{card(pre(chr(10).join(linhas)), "Rastreamento — rodar()")}</div>'
            f'<div style="flex:1;min-width:320px">{card(pre(tabela_txt), "Tabela de transição — tabela()")}</div>'
            '</div>')

    def trocar_modo(*_):
        entrada.value = cfg()["inicio"]
        montar_chips()
        atualizar_gaveta()
        atualizar()

    modo.observe(trocar_modo, names="value")
    entrada.observe(atualizar, names="value")
    montar_chips()
    atualizar_gaveta()
    atualizar()
    return widgets.VBox([modo, entrada, dica, chips, resultado, detalhes, gaveta])

In [4]:
# ---------------------------------------------------------------- aba 2: scanner
PRESETS = {"Entrada válida": sc.ROTEIRO_OK, "Maximal Munch": sc.ROTEIRO_MUNCH,
           "Com erros": sc.ROTEIRO_ERROS, "Roteiro da cancela": conectado.ROTEIRO_CANCELA}
COR_TOKEN = {"RESERVADA": C["amarelo"], "PLACA": C["azul"], "HORA": "#c792ea", "VAGA_ID": "#4dd0e1",
             "IDENT": C["texto"], "INTEIRO": "#ff9e64", "REAL": "#ff9e64", "ATRIB": "#f78cb4",
             "OP_REL": "#f78cb4", "NOVA_LINHA": C["fraco"], "EOF": C["fraco"]}


def html_codigo(src, tokens, erros):
    # código-fonte com cada token colorido e os erros sublinhados em vermelho
    spans = {}
    for t in tokens:
        if t.tipo in ("NOVA_LINHA", "EOF") or not t.lexema:
            continue
        spans.setdefault(t.linha, []).append((t.coluna - 1, t.coluna - 1 + len(t.lexema), t.tipo, False))
    for e in erros:
        spans.setdefault(e.linha, []).append((e.coluna - 1, e.coluna - 1 + len(e.lexema), "ERRO", True))
    saida = []
    for n, texto in enumerate(src.replace("\r", "").split("\n"), 1):
        partes, pos = [], 0
        for c0, c1, tipo, e_erro in sorted(spans.get(n, [])):
            if c0 < pos:
                continue
            partes.append(esc(texto[pos:c0]))
            trecho = esc(texto[c0:c1])
            if e_erro:
                estilo = f'color:{C["erro"]};text-decoration:underline wavy {C["erro"]}'
            else:
                estilo = f'color:{COR_TOKEN.get(tipo, C["texto"])}'
            partes.append(f'<span style="{estilo}">{trecho}</span>')
            pos = c1
        partes.append(f'<span style="color:{C["fraco"]}">{esc(texto[pos:])}</span>')
        saida.append(f'<span style="color:{C["fraco"]}">{n:>3} │ </span>' + "".join(partes))
    return card(f'<pre style="margin:0;background:transparent;color:{C["texto"]};font:13px/1.55 {MONO}">'
                + "\n".join(saida) + "</pre>", "Código com tokens coloridos")


def html_tabela_tokens(tokens, limite=300):
    linhas = ""
    for t in tokens[:limite]:
        cor = COR_TOKEN.get(t.tipo, C["texto"])
        linhas += (f'<tr><td style="padding:2px 12px 2px 0;color:{C["fraco"]}">{t.linha}:{t.coluna}</td>'
                   f'<td style="padding:2px 12px 2px 0;color:{cor};font-weight:bold">{esc(t.tipo)}</td>'
                   f'<td style="padding:2px 0">{esc(t.lexema)}</td></tr>')
    aviso = f'<div style="color:{C["fraco"]};font:11px {UI}">(mostrando {limite} de {len(tokens)})</div>' if len(tokens) > limite else ""
    tabela = (f'<div style="max-height:280px;overflow:auto"><table style="border-collapse:collapse;font:12px {MONO};color:{C["texto"]}">'
              f'<tr style="color:{C["fraco"]};text-align:left"><th style="padding-right:12px">lin:col</th>'
              f'<th style="padding-right:12px">TIPO</th><th>lexema</th></tr>{linhas}</table></div>{aviso}')
    return card(tabela, "Tabela de tokens")


def html_erros(erros):
    if not erros:
        return card(f'<div style="color:{C["ok"]};font:14px {UI}">✔ 0 erros léxicos</div>', "Erros léxicos")
    linhas = ""
    for e in erros:
        linhas += (f'<div style="margin-bottom:8px;font:12px {MONO}">'
                   f'<span style="color:{C["erro"]};font-weight:bold">ERRO {e.linha}:{e.coluna}</span> '
                   f'<span style="color:{C["texto"]}">{esc(e.lexema)}</span> — {esc(e.mensagem)}'
                   f'<div style="color:{C["amarelo"]};margin-left:14px">dica → {esc(e.dica)}</div></div>')
    return card(linhas, f"Erros léxicos ({len(erros)}) — o scanner continua depois de cada erro")


def painel_scanner():
    preset = widgets.Dropdown(options=list(PRESETS), value="Entrada válida", description="Exemplo:")
    editor = widgets.Textarea(value=PRESETS["Entrada válida"],
                              layout=widgets.Layout(width="100%", height="190px"))
    legenda = widgets.HTML("".join(chip(t, c) for t, c in COR_TOKEN.items() if t not in ("NOVA_LINHA", "EOF"))
                           + chip("erro", C["erro"]))
    resumo = widgets.HTML()
    codigo = widgets.HTML()
    tabelas = widgets.HTML()

    def analisar(*_):
        src = editor.value
        tokens, erros = scan(src)
        cor = C["ok"] if not erros else C["erro"]
        resumo.value = (f'<div style="font:14px {UI};color:{cor}"><b>{len(tokens)}</b> tokens, '
                        f'<b>{len(erros)}</b> erro(s) léxico(s)</div>')
        codigo.value = html_codigo(src, tokens, erros)
        tabelas.value = (f'<div style="display:flex;gap:12px;flex-wrap:wrap">'
                         f'<div style="flex:1;min-width:300px">{html_tabela_tokens(tokens)}</div>'
                         f'<div style="flex:1;min-width:300px">{html_erros(erros)}</div></div>')

    preset.observe(lambda _c: setattr(editor, "value", PRESETS[preset.value]), names="value")
    editor.observe(analisar, names="value")
    analisar()
    return widgets.VBox([preset, editor, legenda, resumo, codigo, tabelas])

In [5]:
# ---------------------------------------------------------------- aba 3: cancela
EVENTOS = sorted({ev for (_, ev) in cn.TRANSICOES_CANCELA})
ORDEM = ["FECHADA", "ABRINDO", "ABERTA", "FECHANDO", "TRAVADA"]
ORDEM += [e for e in cn.cancela.estados if e not in ORDEM]


def painel_cancela():
    m = cn.cancela
    m.reiniciar()
    log = []
    estado_html, log_html, roteiro_html = widgets.HTML(), widgets.HTML(), widgets.HTML()
    botoes = []

    def desenhar_estado():
        caixas = ""
        for e in ORDEM:
            if e == m.estado:
                est = f'border:2px solid {C["amarelo"]};background:{C["amarelo"]};color:#1a1d23'
            else:
                est = f'border:1px solid {C["linha"]};background:{C["painel"]};color:{C["fraco"]}'
            caixas += f'<span style="display:inline-block;margin:0 8px 6px 0;padding:8px 14px;border-radius:6px;font:bold 13px {MONO};{est}">{esc(e)}</span>'
        permitidos = m.permitidos()
        caixas += (f'<div style="color:{C["fraco"]};font:12px {UI}">eventos permitidos agora: '
                   f'<span style="color:{C["ok"]}">{esc(", ".join(permitidos))}</span></div>')
        estado_html.value = card(caixas, "Estado da cancela")
        for b in botoes:
            b.button_style = "success" if b.description in permitidos else ""
        if log:
            corpo = "".join(f'<div style="color:{C["ok"] if ok else C["erro"]};font:12px {MONO}">{"✔" if ok else "✘"} {esc(t)}</div>'
                            for ok, t in log[-12:])
        else:
            corpo = f'<div style="color:{C["fraco"]};font:12px {UI}">clique num evento para disparar</div>'
        log_html.value = card(corpo, "Histórico")

    def disparar(evento):
        antes = m.estado
        if m.disparar(evento, imprimir=False):
            log.append((True, f"{antes} —{evento}→ {m.estado}"))
        else:
            log.append((False, f"evento '{evento}' RECUSADO em {antes}. Eventos permitidos: {m.permitidos()}"))
        desenhar_estado()

    for ev in EVENTOS:
        b = widgets.Button(description=ev, layout=widgets.Layout(width="auto"))
        b.on_click(lambda _b, ev=ev: disparar(ev))
        botoes.append(b)
    reset = widgets.Button(description="↻ reiniciar", button_style="warning", layout=widgets.Layout(width="auto"))

    def reiniciar(_b=None):
        m.reiniciar()
        log.clear()
        desenhar_estado()
    reset.on_click(reiniciar)

    roteiro = widgets.Textarea(value=conectado.ROTEIRO_CANCELA, layout=widgets.Layout(width="100%", height="170px"))
    rodar = widgets.Button(description="▶ Executar roteiro na cancela", button_style="primary",
                           layout=widgets.Layout(width="auto"))

    def executar(_b=None):
        texto, _ = capturar(conectado.executar_roteiro, roteiro.value)
        corpo = ""
        for ln in texto.splitlines():
            cor = C["ok"] if "✔" in ln else C["erro"] if "✘" in ln else C["amarelo"] if "⚠" in ln else C["texto"]
            corpo += f'<div style="color:{cor}">{esc(ln) or "&nbsp;"}</div>'
        roteiro_html.value = card(f'<div style="font:12px/1.5 {MONO};white-space:pre;max-height:320px;overflow:auto">{corpo}</div>',
                                  "Tudo conectado: roteiro → scanner → tokens → eventos")
        desenhar_estado()
    rodar.on_click(executar)

    desenhar_estado()
    return widgets.VBox([
        estado_html,
        widgets.HBox(botoes + [reset], layout=widgets.Layout(flex_flow="row wrap")),
        log_html,
        widgets.HTML(f'<div style="font:600 13px {UI};margin-top:10px">Roteiro de comandos (usa o scanner)</div>'),
        roteiro, rodar, roteiro_html])

In [6]:
# ---------------------------------------------------------------- aba 4: checklist
def painel_checklist():
    _, d_placa = capturar(bateria, afd_placa, RE_PLACA, PLACA_ACEITAS, PLACA_REJEITADAS)
    _, d_hora = capturar(bateria, afd_hora, RE_HORA, HORA_ACEITAS, HORA_REJEITADAS)
    tipos = {t.tipo for t in scan(sc.ROTEIRO_OK + "ALERTA x <= 3")[0]}
    toks_err, erros = scan(sc.ROTEIRO_ERROS)
    toks_munch, _ = scan(sc.ROTEIRO_MUNCH)
    lexemas_munch = {t.lexema for t in toks_munch}
    m = cn.cancela
    checks = {
        "AFD_PLACA ≥ 5 estados": len(afd_placa.estados) >= 5,
        "AFD_HORARIO ≥ 5 estados": len(afd_hora.estados) >= 5,
        "0 divergências AFD × RegEx": d_placa == 0 and d_hora == 0,
        "palavra vazia testada": "" in PLACA_REJEITADAS and "" in HORA_REJEITADAS,
        "máquina ≥ 5 estados e ≥ 6 transições": len(m.estados) >= 5 and len(m.transicoes) >= 6,
        "scanner ≥ 8 tipos de token": len(tipos - {"NOVA_LINHA", "EOF"}) >= 8,
        "≥ 3 palavras reservadas": len(sc.RESERVADAS) >= 3,
        "Maximal Munch (<=, == e 12.00 lidos como um token só)": {"<=", "==", "12.00"} <= lexemas_munch,
        "erros com linha, coluna e dica": all(e.linha and e.coluna and e.dica for e in erros),
        "scanner continua após erro": any(t.linha == 4 for t in toks_err),
        "AFDs mínimos (bônus)": not pares_equivalentes(afd_placa) and not pares_equivalentes(afd_hora),
        "Scanner por tabela ≡ scanner com while (bônus)": not sc.bateria_scanner(),
    }
    linhas = ""
    for nome, ok in checks.items():
        linhas += (f'<div style="color:{C["ok"] if ok else C["erro"]};font:14px/1.9 {UI}">'
                   f'{"✔" if ok else "✘"} {esc(nome)}</div>')
    total = sum(checks.values())
    cab = f'<div style="font:bold 16px {UI};margin-bottom:8px">{total} de {len(checks)} itens ok</div>'
    return widgets.VBox([widgets.HTML(card(cab + linhas, "Checklist dos requisitos"))])

In [7]:
# ---------------------------------------------------------------- Painel
tabs = widgets.Tab(children=[painel_validadores(), painel_scanner(), painel_cancela(), painel_checklist()])
for i, titulo in enumerate(["Validadores", "Scanner", "Cancela", "Checklist"]):
    tabs.set_title(i, titulo)

display(HTML(f'<div style="font:bold 22px {UI};margin:4px 0 8px">'
             f'<span style="background:{C["amarelo"]};color:#1a1d23;padding:0 9px;border-radius:5px;margin-right:10px">P</span>'
             'Estacionamento Inteligente</div>'))
display(tabs)

## Saída em texto (R1–R5)

As células abaixo rodam as demonstrações do projeto no formato de texto: `tabela()`, `rodar()`, bateria AFD × RegEx, máquina de estados, scanner e o roteiro conectado.

In [8]:
# R1–R3: AFDs (tabela(), rodar() e bateria AFD × RegEx)
divergencias = validadores.demo()
print("divergências:", divergencias)

AFD 1 — PLACA (antiga + Mercosul)
Tabela de transição — AFD_PLACA
estado     L      N      -   
-----------------------------
→ p0       p1     —      —      início
  p1       p2     —      —      1 letra
  p2       p3     —      —      2 letras
  p3       —      p6     p4     3 letras
  p4       —      p5     —      antiga: leu '-'
  p5       —      p7     —      antiga: 1º dígito
  p6       p7     —      —      Mercosul: 1º dígito
  p7       —      p8     —      faltam 2 dígitos
  p8       —      p9     —      falta 1 dígito
 *p9       —      —      —      placa completa
(→ inicial, * final, — sem transição = morre)

--- R2: rastreamento passo a passo ---
AFD_PLACA: rodar('BRA2E19')
  início em p0
  [0] 'B'   classe L          δ(p0, L) = p1
  [1] 'R'   classe L          δ(p1, L) = p2
  [2] 'A'   classe L          δ(p2, L) = p3
  [3] '2'   classe N          δ(p3, N) = p6
  [4] 'E'   classe L          δ(p6, L) = p7
  [5] '1'   classe N          δ(p7, N) = p8
  [6] '9'   classe N       

In [9]:
# R4: máquina de estados da cancela (2 fluxos válidos e 2 inválidos)
cn.demo()
cn.cancela.reiniciar()

MÁQUINA DE ESTADOS — CANCELA
5 estados: ['ABERTA', 'ABRINDO', 'FECHADA', 'FECHANDO', 'TRAVADA']
9 transições
  ABERTA    permite ['tempo_esgotado', 'veiculo_passou']
  ABRINDO   permite ['falha_motor', 'fim_abertura']
  FECHADA   permite ['liberar']
  FECHANDO  permite ['falha_motor', 'fim_fechamento', 'sensor_obstaculo']
  TRAVADA   permite ['manutencao_ok']

▶ Válido 1 — entrada normal
  ✔ FECHADA   --liberar--> ABRINDO
  ✔ ABRINDO   --fim_abertura--> ABERTA
  ✔ ABERTA    --veiculo_passou--> FECHANDO
  ✔ FECHANDO  --fim_fechamento--> FECHADA
  FLUXO VÁLIDO (terminou em FECHADA)

▶ Válido 2 — obstáculo reabre a cancela
  ✔ FECHADA   --liberar--> ABRINDO
  ✔ ABRINDO   --fim_abertura--> ABERTA
  ✔ ABERTA    --veiculo_passou--> FECHANDO
  ✔ FECHANDO  --sensor_obstaculo--> ABRINDO
  ✔ ABRINDO   --fim_abertura--> ABERTA
  ✔ ABERTA    --tempo_esgotado--> FECHANDO
  ✔ FECHANDO  --fim_fechamento--> FECHADA
  FLUXO VÁLIDO (terminou em FECHADA)

▶ Inválido 1 — veículo passou com a cancela fecha

In [10]:
# R5 + bônus: mini-scanner (tokens, Maximal Munch, erros com dica) e o scanner por tabela (δ combinado)
_ = sc.demo()

SCANNER — entrada válida
lin:col TIPO        lexema
2:1     RESERVADA   ENTRADA
2:9     PLACA       BRA2E19
2:17    HORA        08:15
2:23    RESERVADA   VAGA
2:28    VAGA_ID     A12
2:31    NOVA_LINHA  \n
3:1     RESERVADA   SAIDA
3:7     PLACA       BRA2E19
3:15    HORA        10:40
3:21    RESERVADA   PAGO
3:26    REAL        12.00
3:50    NOVA_LINHA  \n
4:1     RESERVADA   ENTRADA
4:9     PLACA       ABC-1234
4:18    HORA        09:00
4:24    RESERVADA   VAGA
4:29    VAGA_ID     B7
4:31    NOVA_LINHA  \n
5:1     RESERVADA   TARIFA
5:8     IDENT       valor_hora
5:19    ATRIB       =
5:21    REAL        6.00
5:25    NOVA_LINHA  \n
6:1     RESERVADA   ALERTA
6:8     IDENT       permanencia
6:20    OP_REL      >=
6:23    INTEIRO     240
6:26    NOVA_LINHA  \n
7:1     RESERVADA   ALERTA
7:8     IDENT       vagas_livres
7:21    OP_REL      <=
7:24    INTEIRO     5
7:25    NOVA_LINHA  \n
8:1     EOF         

34 tokens, 0 erro(s) léxico(s)

SCANNER — Maximal Munch
lin:col TIPO        lex

Bateria scanner por tabela × scanner com while: 2007 entradas, 0 divergência(s)


0

In [11]:
# Bônus: tudo conectado (roteiro → scanner → eventos da cancela)
conectado.demo()
cn.cancela.reiniciar()

BÔNUS — TUDO CONECTADO (roteiro → eventos da cancela)
cancela começa em FECHADA
linha 1: ENTRADA BRA2E19 08:15 VAGA A12
  ✔ FECHADA   --liberar--> ABRINDO
linha 2: ABRIU
  ✔ ABRINDO   --fim_abertura--> ABERTA
linha 3: PASSOU
  ✔ ABERTA    --veiculo_passou--> FECHANDO
linha 4: OBSTACULO
  ✔ FECHANDO  --sensor_obstaculo--> ABRINDO
linha 5: ABRIU
  ✔ ABRINDO   --fim_abertura--> ABERTA
linha 6: TEMPO
  ✔ ABERTA    --tempo_esgotado--> FECHANDO
linha 7: FECHOU
  ✔ FECHANDO  --fim_fechamento--> FECHADA
linha 8: SAIDA ABC-1234 11:00
  ⚠ pagamento pendente: nenhum evento gerado (cancela continua FECHADA)
linha 9: PASSOU
  ✘ evento 'veiculo_passou' RECUSADO em FECHADA. Eventos permitidos: ['liberar']
linha 10: SAIDA BRA2E19 10:40 PAGO 12.00
  ✔ FECHADA   --liberar--> ABRINDO
linha 11: ABRIU
  ✔ ABRINDO   --fim_abertura--> ABERTA
linha 12: PASSOU
  ✔ ABERTA    --veiculo_passou--> FECHANDO
linha 13: FECHOU
  ✔ FECHANDO  --fim_fechamento--> FECHADA
cancela termina em FECHADA
